# A tour of the notebook display

This notebook drives a **simulated** Hamilton STARlet (no hardware, no server) so you can see
how Praxis draws labware, records runs and explains errors.

Run the cells **one at a time from the top** with Shift+Enter. Each section relies on the state
the cells before it leave behind, and the last section raises errors on purpose, so *Run All*
stops at the first of them.

Your edits to this file can be overwritten by the next build. To keep a copy, use
*File → Save Notebook As…* first.

## 1. Build a deck

A STARlet deck with one 300 µL tip rack and two 96-well plates: `source` (200 µL in every well)
and `assay` (empty). Tip and volume tracking are on, so the display can show what is in each
well and which tips are gone.

In [ ]:
from pylabrobot.legacy.liquid_handling import LiquidHandler
from pylabrobot.legacy.liquid_handling.backends import LiquidHandlerChatterboxBackend
from pylabrobot.resources import set_tip_tracking, set_volume_tracking
from pylabrobot.resources.corning import cor_96_wellplate_360uL_Fb
from pylabrobot.resources.hamilton import (
    STARLetDeck,
    hamilton_96_tiprack_300uL_filter,
    hamilton_plate_carrier_L5_ac,
    hamilton_tip_carrier_L5,
)

set_tip_tracking(True)
set_volume_tracking(True)

deck = STARLetDeck()
lh = LiquidHandler(backend=LiquidHandlerChatterboxBackend(num_channels=8), deck=deck)
await lh.setup()

tip_car = hamilton_tip_carrier_L5(name="tip_carrier")
tip_car[0] = hamilton_96_tiprack_300uL_filter(name="tips_300")
deck.assign_child_resource(tip_car, track=3)

plate_car = hamilton_plate_carrier_L5_ac(name="plate_carrier")
plate_car[0] = source = cor_96_wellplate_360uL_Fb(name="source")
plate_car[1] = assay = cor_96_wellplate_360uL_Fb(name="assay")
deck.assign_child_resource(plate_car, track=9)

tips = deck.get_resource("tips_300")
for well in source.get_all_items():
    well.tracker.set_volume(200.0)

## 2. Draw labware

Ending a cell with a plate, a tip rack or the deck draws it: wells filled to their volume,
tips present or gone, and the deck to scale with its carriers on their rails, each with a
one-line summary underneath.

In [ ]:
source

In [ ]:
tips

In [ ]:
deck

In [ ]:
assay

## 3. Record a run

`RunLedger` records every liquid-handling step inside its `with` block and, when the block
ends, shows them as a table (steps, tip cycles, µL moved) followed by the plates it dispensed
into, with the wells that changed outlined.

This run moves 50, 100 and 150 µL into columns 1–3 of `assay`. When it finishes, the empty
`assay` drawing from section 2 is marked out of date, because the plate it shows has changed.

The plain text that prints first is the simulated backend's own log of each command; the
ledger table comes after it.

In [ ]:
from praxis.display import RunLedger

with RunLedger(lh):
    for col, vol in ((1, 50.0), (2, 100.0), (3, 150.0)):
        await lh.pick_up_tips(tips[f"A{col}:H{col}"])
        await lh.aspirate(source[f"A{col}:H{col}"], vols=[vol] * 8)
        await lh.dispense(assay[f"A{col}:H{col}"], vols=[vol] * 8)
        await lh.discard_tips()

In [ ]:
assay

In [ ]:
tips

## 4. Errors, explained

Each cell below fails on purpose. Instead of a bare traceback, the error panel says what went
wrong, on which wells and channels, and what to check (the full traceback is still one click
away).

First, pick up fresh tips and take 10 µL from `source` so the channels hold something.

In [ ]:
await lh.pick_up_tips(tips["A4:H4"])

In [ ]:
await lh.aspirate(source["A1:H1"], vols=[10.0] * 8)

**Not enough liquid.** `assay` column 1 holds 50 µL; this asks for 80.

In [ ]:
await lh.aspirate(assay["A1:H1"], vols=[80.0] * 8)

**Too much liquid for the well.** The first line forces channel 0's tip to hold 400 µL
(`set_volume` does not check the tip's capacity); dispensing it into A2, which already holds
100 µL of its 360, overflows the well.

In [ ]:
lh.head[0].get_tip().tracker.set_volume(400)  # past max_volume on purpose: set_volume does not validate it
await lh.dispense([assay.get_item("A2")], vols=[400.0], use_channels=[0])

**Tips already on.** The channels still hold the column 4 tips.

In [ ]:
await lh.pick_up_tips(tips["A5:H5"])

**No tip.** Drop the tips, then try to aspirate on channel 0 without one.

In [ ]:
await lh.discard_tips()
await lh.aspirate([source.get_item("A1")], vols=[50.0], use_channels=[0])

**The 96-head.** Put a second rack on the tip carrier and pick it up with the 96-head
(this cell succeeds).

In [ ]:
tip_car[1] = tips_96 = hamilton_96_tiprack_300uL_filter(name="tips_96")
await lh.pick_up_tips96(tips_96)

Then aspirate 50 µL from every well of `assay`. Columns 4–12 are empty, so this fails.
The panel names the empty wells and adds a note: PyLabRobot still records moves that did not
happen, so check the drawings before trusting the plate's state.

In [ ]:
await lh.aspirate96(assay, volume=50.0)

## Where to go next

Create a notebook of your own from the file browser and copy section 1 to start. Change a
volume and re-run a drawing cell to see the older drawing marked out of date.